In [2]:
import pandas as pd
import math as m
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import OneHotEncoder,StandardScaler

In [3]:
df=pd.read_csv('diamonds.csv')
df.head()

,Unnamed: 0,carat,cut,color,clarity,depth,table,price,x,y,z
0,1,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,2,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,3,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,4,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,5,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53940 entries, 0 to 53939
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  53940 non-null  int64  
 1   carat       53940 non-null  float64
 2   cut         53940 non-null  object 
 3   color       53940 non-null  object 
 4   clarity     53940 non-null  object 
 5   depth       53940 non-null  float64
 6   table       53940 non-null  float64
 7   price       53940 non-null  int64  
 8   x           53940 non-null  float64
 9   y           53940 non-null  float64
 10  z           53940 non-null  float64
dtypes: float64(6), int64(2), object(3)
memory usage: 4.5+ MB


In [5]:
df.drop('Unnamed: 0',axis=1,inplace=True)

In [6]:
x=df.drop(columns=['price','cut','color','clarity'],axis=1)
y=df['price']

In [7]:
xtrain,xtest,ytrain,ytest=train_test_split(x,y,train_size=0.8,random_state=42)

In [8]:
k=m.sqrt(53940)
k

232.24986544667792

In [9]:
param_selection=GridSearchCV(estimator=KNeighborsRegressor(),param_grid={
'n_neighbors':[5,232,230,235,229],'weights':['uniform','distance'],'metric':['euclidean','manhattan']},cv=5,scoring='r2')

In [10]:
param_selection.fit(xtrain,ytrain)

GridSearchCV(cv=5, estimator=KNeighborsRegressor(),
             param_grid={'metric': ['euclidean', 'manhattan'],
                         'n_neighbors': [5, 232, 230, 235, 229],
                         'weights': ['uniform', 'distance']},
             scoring='r2')

In [11]:
model=param_selection.best_params_
model

{'metric': 'manhattan', 'n_neighbors': 229, 'weights': 'distance'}

In [12]:
pd.DataFrame(param_selection.cv_results_).sort_values('rank_test_score')

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_metric,param_n_neighbors,param_weights,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
19,0.071611,0.026973,0.904051,0.453235,manhattan,229,distance,"{'metric': 'manhattan', 'n_neighbors': 229, 'w...",0.880946,0.878126,0.879398,0.884403,0.883851,0.881345,0.002447,1
15,0.056618,0.009635,0.631406,0.072428,manhattan,230,distance,"{'metric': 'manhattan', 'n_neighbors': 230, 'w...",0.880936,0.878098,0.879392,0.884387,0.883845,0.881331,0.002451,2
13,0.055011,0.004062,0.594960,0.015500,manhattan,232,distance,"{'metric': 'manhattan', 'n_neighbors': 232, 'w...",0.880930,0.878075,0.879366,0.884369,0.883822,0.881312,0.002452,3
17,0.051801,0.003620,0.599818,0.010603,manhattan,235,distance,"{'metric': 'manhattan', 'n_neighbors': 235, 'w...",0.880957,0.878045,0.879318,0.884374,0.883824,0.881304,0.002468,4
18,0.054959,0.003510,0.561620,0.013485,manhattan,229,uniform,"{'metric': 'manhattan', 'n_neighbors': 229, 'w...",0.879198,0.875199,0.877531,0.882646,0.882643,0.879443,0.002906,5
14,0.053400,0.003758,0.568156,0.014158,manhattan,230,uniform,"{'metric': 'manhattan', 'n_neighbors': 230, 'w...",0.879178,0.875150,0.877528,0.882624,0.882629,0.879422,0.002913,6
12,0.055009,0.003973,0.568335,0.012311,manhattan,232,uniform,"{'metric': 'manhattan', 'n_neighbors': 232, 'w...",0.879167,0.875119,0.877501,0.882601,0.882588,0.879395,0.002912,7
16,0.058360,0.005358,0.581478,0.028241,manhattan,235,uniform,"{'metric': 'manhattan', 'n_neighbors': 235, 'w...",0.879219,0.875064,0.877420,0.882609,0.882596,0.879382,0.002941,8
9,0.053484,0.003920,0.601715,0.006309,euclidean,229,distance,"{'metric': 'euclidean', 'n_neighbors': 229, 'w...",0.875937,0.872721,0.872855,0.877931,0.877338,0.875356,0.002195,9
5,0.056613,0.006518,0.596529,0.012426,euclidean,230,distance,"{'metric': 'euclidean', 'n_neighbors': 230, 'w...",0.875927,0.872681,0.872847,0.877891,0.877298,0.875329,0.002190,10


In [13]:
x=df.drop('price',axis=1)
y=df['price']

In [14]:
pred_train=param_selection.predict(xtrain)

In [15]:
num_col=xtrain.select_dtypes(include='number').columns
cat_col=xtrain.select_dtypes(include='object').columns

In [16]:
xtrain,xtest,ytrain,ytest=train_test_split(x,y,train_size=0.8,random_state=42)

In [17]:
preprocessing=ColumnTransformer(transformers=[
    ('encoder',OneHotEncoder(dtype='int',sparse_output=False,handle_unknown='ignore'),cat_col),
    ('scaler',StandardScaler(),num_col)
])
preprocessing.fit_transform(xtrain,ytrain)

array([[ 2.56005606, -2.55074762,  2.93386055,  2.22945022,  2.13820916,
         1.73820671],
       [ 0.44739205, -1.22042647,  1.13957453,  0.74754991,  0.65671016,
         0.5377332 ],
       [ 0.63753181,  0.52999608,  0.24243153,  0.76540413,  0.70028366,
         0.79195112],
       ...,
       [-0.98921948, -1.01037577,  0.24243153, -1.10928903, -1.11237394,
        -1.18529936],
       [ 0.21499901,  0.74004679,  0.69100303,  0.35475706,  0.25583396,
         0.39650102],
       [ 0.72203837, -0.94035886,  0.24243153,  0.97072767,  0.91815116,
         0.80607434]])

In [18]:
preprocessing.transform(xtest)

array([[-1.17935924,  0.24992847, -0.65471148, -1.57349876, -1.51325014,
        -1.51013337],
       [-0.46105347, -1.22042647, -0.20613998, -0.26121355, -0.27576274,
        -0.39439917],
       [-0.841333  ,  0.24992847, -1.10328299, -0.86825705, -0.86836234,
        -0.83221892],
       ...,
       [-1.03147276, -2.62076452,  2.03671754, -1.1717788 , -1.10365924,
        -1.35477797],
       [ 0.91217813,  0.52999608, -1.55185449,  0.997509  ,  0.94429526,
         1.03204582],
       [ 0.59527853,  1.0201144 ,  0.69100303,  0.59578903,  0.72642776,
         0.79195112]])

In [19]:
pipeline=Pipeline(steps=[
    ('preprocess',preprocessing),
    ('model',KNeighborsRegressor())
])

pipeline.fit(xtrain,ytrain)

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('encoder',
                                                  OneHotEncoder(dtype='int',
                                                                handle_unknown='ignore',
                                                                sparse_output=False),
                                                  Index([], dtype='object')),
                                                 ('scaler', StandardScaler(),
                                                  Index(['carat', 'depth', 'table', 'x', 'y', 'z'], dtype='object'))])),
                ('model', KNeighborsRegressor())])

In [20]:
param=GridSearchCV(estimator=pipeline,param_grid={
'model__n_neighbors':[5,232,230,235,229],'model__weights':['uniform','distance'],'model__metric':['euclidean','manhattan']},cv=5,scoring='r2')

In [21]:
param.fit(xtrain,ytrain)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocess',
                                        ColumnTransformer(transformers=[('encoder',
                                                                         OneHotEncoder(dtype='int',
                                                                                       handle_unknown='ignore',
                                                                                       sparse_output=False),
                                                                         Index([], dtype='object')),
                                                                        ('scaler',
                                                                         StandardScaler(),
                                                                         Index(['carat', 'depth', 'table', 'x', 'y', 'z'], dtype='object'))])),
                                       ('model', KNeighborsRegressor())]),
             param_grid={'model__metric': ['euclidean', 'manhattan'],
                         'model__n_neighbors': [5, 232, 230, 235, 229],
                         'model__weights': ['uniform', 'distance']},
             scoring='r2')

In [22]:
ypred_xtrain=pipeline.predict(xtrain)
ypred_xtest=pipeline.predict(xtest)

In [23]:
r2_score(ytrain,pred_train)

0.9988898574859044

In [24]:
r2_xtrain=r2_score(ytrain,ypred_xtrain)
r2_xtest=r2_score(ytest,ypred_xtest)

In [25]:
print("xtrain:",r2_xtrain,"\nxtest:",r2_xtest)

xtrain: 0.9127026060124375 
xtest: 0.8678447914310016
